# 데이터 전처리

- 처리가 필요한 데이터
    + 누락 데이터 (NA) -> 누락 데이터 처리
    + 타입이 올바르지 않은 데이터 -> 데이터 타입 변경
    + 콤마, 공백, 모호한 대/소문자 사용 -> 콤마, 공백 제거, 대/소문자 변경
    + 의미없는 데이터 -> 의미있는 데이터 선별 및 제거

## NA의 종류

- NA == Not Available == Missing Data
- None, np.nan, pd.NaT
    + np.nan : float 타입으로 숫자의 missing 을 의미
    + pd.NaT : np.datetime64 타입으로 날짜의 missing 을 의미
    + np.inf : pd.options.mode.use_inf_as_na = True 를 하면 missing data 취급 / 기본값은 False 임
- pandas 에서의 연산은 기본적으로 NA 데이터를 제외하고 처리됨

In [1]:
import pandas as pd
import numpy as np
import shelve

In [2]:
pd.options.mode.use_inf_as_na = True

s = pd.Series([np.nan, pd.NaT, None, np.inf])
display(s)

print(s.count(), s.sum())

0     NaN
1     NaT
2    None
3     NaN
dtype: object

0 0


In [3]:
pd.options.mode.use_inf_as_na = False

s = pd.Series([np.nan, pd.NaT, None, np.inf])
display(s)

print(s.count(), s.sum())

0     NaN
1     NaT
2    None
3     inf
dtype: object

1 inf


## NA 데이터의 상등 비교

- None, np.inf 는 상등 비교가 가능함
- np.nan, pd.NaT는 상등 비교가 불가능함 (isna() 함수를 사용함)

In [4]:
print(f'None :   {None == None}',
      f'np.inf : {np.inf == np.inf}',
      f'np.nan : {np.nan == np.nan}',
      f'pd.NaT : {pd.NaT == pd.NaT}', sep="\n")

None :   True
np.inf : True
np.nan : False
pd.NaT : False


In [5]:
pd.options.mode.use_inf_as_na = True
s = pd.Series([np.nan, pd.NaT, None, np.inf])
s.isna()

0    True
1    True
2    True
3    True
dtype: bool

In [6]:
pd.options.mode.use_inf_as_na = False
s = pd.Series([np.nan, pd.NaT, None, np.inf])
s.isna()

0     True
1     True
2     True
3    False
dtype: bool

## NA 정보 확인

- `df.info()`
    + index, columns, dtypes, memory usage 정보 출력
    + 출력 정보의 정도를 조절할 수 있는 parameter 가 있음
    + memory_usage='deep', deep memory introspection 설정
    - https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html#pandas-dataframe-info
- `df.isna()`
    + Boolean 데이터로 작성된 DataFrame 객체 반환 (NA value -> True)
    + isna, isnull 은 동일 동작 (isnull 은 isna 의 alias)
    + any(), all() 등으로 정보를 요약할 수 있음
    + `df.isna().any()`
    + `df.isna().all()`
    + `df.isna().any().any()` : DataFrame 에 하나라도 NA가 있으면 True
    + https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html#pandas-dataframe-isna
- `df.any(axis=0)`
    + column 별로 값이 하나라도 True 인 경우 True 아니면 False
    + axis=1 로 하면 row 별로 확인
- `df.all(axis=0)`
    + column 별로 모든 값이 True 인 경우 True 아니면 False
    + axis=1 로 하면 row 별로 확인    

In [7]:
df = pd.read_csv('./data/easySample2.csv')
df.head()

,ID,pname,birth,dept,english,japanese,chinese,salary,overtime
0,18030201,James Kim,1990-01-23,Education,1,1.0,,"3,456",23:10:10
1,18030202,Rose Hwang,1992-10-11,Marketing,,2.0,NaN,"4,320",10:15:17
2,19030401,Sam Park,1995-07-02,Education,1,NaN,NaN,"5,600",16:21:10
3,19070101,Chris Jang,1990-11-23,Education,NaN,NaN,3,"4,500",15:00:20
4,19070102,Grace Lee,1993-02-01,Marketing,NaN,NaN,NaN,"3,150",21:19:50


In [8]:
df.info()
# 하기 정보 중 RangeIndex: 10 entries, 0 to 9 을 통하여 전체 10개의 Entry 가 있음을 알 수 있음
# 4   english   6 non-null      object 
# 5   japanese  5 non-null      float64
# 6   chinese   6 non-null      object 
# 상기 3개의 Column 의 경우 10개의 데이터가 안되므로 NA 데이터가 있다고 판단할 수 있음
# 특히 english 의 경우 object dtype 이므로 데이터 중 공백이 포함되어 있음을 식별할 수 있음

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        10 non-null     int64  
 1   pname     10 non-null     object 
 2   birth     10 non-null     object 
 3   dept      10 non-null     object 
 4   english   6 non-null      object 
 5   japanese  5 non-null      float64
 6   chinese   6 non-null      object 
 7   salary    10 non-null     object 
 8   overtime  10 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 848.0+ bytes


In [9]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        10 non-null     int64  
 1   pname     10 non-null     object 
 2   birth     10 non-null     object 
 3   dept      10 non-null     object 
 4   english   6 non-null      object 
 5   japanese  5 non-null      float64
 6   chinese   6 non-null      object 
 7   salary    10 non-null     object 
 8   overtime  10 non-null     object 
dtypes: float64(1), int64(1), object(7)
memory usage: 4.4 KB


In [10]:
df1 = df[['english', 'japanese', 'chinese']]
df1.isna()

,english,japanese,chinese
0,False,False,False
1,False,False,True
2,False,True,True
3,True,True,False
4,True,True,True
5,True,True,False
6,False,True,True
7,False,False,False
8,False,False,False
9,True,False,False


In [11]:
df1.isna().any()

english     True
japanese    True
chinese     True
dtype: bool

In [12]:
df1.isna().any(axis=1)

0    False
1     True
2     True
3     True
4     True
5     True
6     True
7    False
8    False
9     True
dtype: bool

In [13]:
df1.isna().all()

english     False
japanese    False
chinese     False
dtype: bool

In [14]:
df1.isna().all(axis=1)

0    False
1    False
2    False
3    False
4     True
5    False
6    False
7    False
8    False
9    False
dtype: bool

In [15]:
df1.isna().any().any()

True

## NA 제거

- `df.dropna(axis=0, how='any', thresh=None, subset=None, inplace=False)`
- axis
    + axis=0 이거나 'index' : NA value 포함 행(row) 제거
    + axis=1 이거나 'columns' : NA value 포함 열(column) 제거    
- how
    + how='any' : NA value 가 하나라도 포함된 경우 True
    + how='all' : 모든 값이 NA value 인 경우 True    
- thresh
    + int, non-NA value 갯수가 설정 값 이상일 떄 제거 안 함
- subset
    + array-like, NA value 를 살펴볼 label 목록
    + axis=0 : columns 에 대한 label 을 목록으로 작성함
- inplace
    + bool, True 인 경우 대상에 직접 반영하고, None 을 반환

In [16]:
df = pd.read_csv('./data/easySample2.csv')
df.head()

,ID,pname,birth,dept,english,japanese,chinese,salary,overtime
0,18030201,James Kim,1990-01-23,Education,1,1.0,,"3,456",23:10:10
1,18030202,Rose Hwang,1992-10-11,Marketing,,2.0,NaN,"4,320",10:15:17
2,19030401,Sam Park,1995-07-02,Education,1,NaN,NaN,"5,600",16:21:10
3,19070101,Chris Jang,1990-11-23,Education,NaN,NaN,3,"4,500",15:00:20
4,19070102,Grace Lee,1993-02-01,Marketing,NaN,NaN,NaN,"3,150",21:19:50


In [17]:
# Regular Expression 을 통하여 White Space 를 np.nan 으로 변경
df = df.replace(r'^\s+$', np.nan, regex=True)

In [18]:
df = df.iloc[:3, 3:7]
df

,dept,english,japanese,chinese
0,Education,1,1.0,NaN
1,Marketing,NaN,2.0,NaN
2,Education,1,NaN,NaN


In [19]:
# NA 가 아닌 것을 확인 == ~df.isna()
df.notna()

,dept,english,japanese,chinese
0,True,True,True,False
1,True,False,True,False
2,True,True,False,False


In [20]:
# NA 가 하나라도 포함된 열 제거
df.dropna(axis=1)

,dept
0,Education
1,Marketing
2,Education


In [21]:
# NA 가 하나라도 포함된 행 제거
df.dropna()

,dept,english,japanese,chinese


In [22]:
# 각 컬럼에 대해 모든 항이 NA 인 경우 제거
df.dropna(axis=1, how='all')

,dept,english,japanese
0,Education,1,1.0
1,Marketing,NaN,2.0
2,Education,1,NaN


In [23]:
# NA 가 아닌 값이 3개 이상인 행에 대해 제거 안 함
df.dropna(thresh=3)

,dept,english,japanese,chinese
0,Education,1,1.0,NaN


In [24]:
# dropna 의 기준을 english 와 japanese 로 제한하며, 1개라도 NA 가 있는 행을 제거
df.dropna(how='any', subset=['english', 'japanese'])

,dept,english,japanese,chinese
0,Education,1,1.0,NaN


In [25]:
# 1개라도 NA 가 있는 행을 제거해서 df 자체를 수정
df.dropna(inplace=True)
df

,dept,english,japanese,chinese


## NA 채우기

- `df.fillna(value=None, method=None, axis=None, inplace=False, limit=None, ...)`
- NA value 를 value 또는 method 를 사용하여 변경한 DataFrame 객체 반환
- value
    + scalar, dit, Series, DataFrame
    + NA values 를 대신할 값을 지정함
    + dict, Series, DataFrame 을 사용해 행/열 별 채우기 값 별도 지정 가능
- method
    + {'backfill', 'bfill', 'pad', 'ffill', None}
    + value= None 일 때, NA values 를 대신할 값 선정 방법을 지정함
    + 'backfill', 'bfill' : [아래 -> 위]로 올라가면서 다음 발견되는 valid observation 으로 채움
    + 'pad', 'ffill' : [위 -> 아래]로 내려가면서 이전에 발견된 valid observation 으로 채움
- asis
    + axis= 0 이거나 'index' : 행 방향으로 채우기 진행
    + axis= 1 이거나 'columns' : 열 방향으로 채우기 진행    
- inplace
    + bool, True 인 경우 대상에 직접 반영하고, None 을 반환
- limit
    + NA values 를 다른 value 로 변경하는 동작의 최대 횟수

In [26]:
df = pd.read_csv("./data/easySample2.csv")
df = df.replace(r'^\s+$', np.nan, regex=True)

# index 는 indexing 에 포함되지 않음
df = df.iloc[:6, [4,6]]
df.iloc[0,1] = 2
df.iloc[1,1] = 3
df

,english,chinese
0,1,2
1,NaN,3
2,1,NaN
3,NaN,3
4,NaN,NaN
5,NaN,1


In [27]:
# 모든 NA 를 0 으로 채움
df.fillna(value=0)

,english,chinese
0,1,2
1,0,3
2,1,0
3,0,3
4,0,0
5,0,1


In [28]:
# english 의 NA 는 -1, chinese 의 NA 는 -2 로 채움
df.fillna(value={"english": -1, "chinese": -2})

,english,chinese
0,1,2
1,-1,3
2,1,-2
3,-1,3
4,-1,-2
5,-1,1


In [29]:
# NA 를 채울 때 이전에 발견된 값으로 채우는데 2개까지만 채움
df.fillna(method='ffill', limit=2)

,english,chinese
0,1,2
1,1,3
2,1,3
3,1,3
4,1,3
5,NaN,1


In [30]:
# NA를 채울 때 다음 발견되는 값으로 채움
df.fillna(method='backfill')

,english,chinese
0,1,2
1,1,3
2,1,3
3,NaN,3
4,NaN,1
5,NaN,1


## Value 대체

- `df.replace(to_replace, value=None, inplace=False, limit=None, regex=False, ...)`
- to_replace 로 주어진 대상이 value 로 주어진 값으로 변경된 DataFrame 객체
- to_replace
    + str, regex, list, dict, Series, int, float, None
    + value 로 대체될 값들을 찾는 방법
    + API 에서 상세 설명 참조
      https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html#pandas-dataframe-replace
- value
    + scalar, dict, list, str, regex, default None
    + to_replace 에 매칭되는 값을 대체할 값을 지정
    + dict 을 사요하여 열 별 채우기 값을 별도 지정 가능
- inplace
    + bool, True 인 경우 대상에 직접 반영하고, None 을 반환
- limit
    + NA values 를 다른 value 로 변경하는 동작의 최대 횟수
- regex
    + bool or same types as to_replace
    + True 설정 시 to_replace 및 value 의 정규식 사용 가능
    + to_replace 는 str 을 사용해야 함

In [31]:
df = pd.read_csv("./data/easySample2.csv")
df = df.replace(r'^\s+$', np.nan, regex=True)
df = df[['dept']]
df

,dept
0,Education
1,Marketing
2,Education
3,Education
4,Marketing
5,Education
6,Accounting
7,Sales
8,Sales
9,Education


In [32]:
# Education 을 E 로 변경
df.replace('Education', value='E')

,dept
0,E
1,Marketing
2,E
3,E
4,Marketing
5,E
6,Accounting
7,Sales
8,Sales
9,E


In [33]:
# Education, Marketing, Sales, Accounting 을 각각 E, M, S, A 로 변경
df.replace(to_replace=['Education', 'Marketing', 'Sales', 'Accounting'], 
           value=['E', 'M', 'S', 'A'])

,dept
0,E
1,M
2,E
3,E
4,M
5,E
6,A
7,S
8,S
9,E


In [34]:
temp = df['dept'].unique()
v = [ x[0] for x in temp ]
df.replace(temp, v)

,dept
0,E
1,M
2,E
3,E
4,M
5,E
6,A
7,S
8,S
9,E


In [35]:
# Education, Marketing, Sales, Accounting 을 각각 0,1,2,3 로 변경
df.replace(to_replace=['Education', 'Marketing', 'Sales', 'Accounting'], 
           value=['E', 'M', 'S', 'A'])

,dept
0,E
1,M
2,E
3,E
4,M
5,E
6,A
7,S
8,S
9,E


In [36]:
# dictionary 사용 시 { to_replace: value } 구조
df.replace({'Education': 0, 'Marketing': 1, 'Sales': 2, 'Accounting': 3})

,dept
0,0
1,1
2,0
3,0
4,1
5,0
6,3
7,2
8,2
9,0


In [37]:
# Series 사용 시 to_replace 가 index 인 구조
dept_list = df['dept'].unique()
value_list = np.array(range(len(dept_list)))
s1 = pd.Series(value_list, index=dept_list)

df.replace(s1)

,dept
0,0
1,1
2,0
3,0
4,1
5,0
6,2
7,3
8,3
9,0


# DataFrame 합치기

## pd.concat

- 여러 개의 데이터프레임 하나로 합치기
- `pd.concat(objs, axis=0, join='outer', ignore_index=False, verify_integrity=False, ...)`
- index 를 기준으로 행/열 방향으로 DataFrame 을 병합
- objs : a sequence or mapping of Series or DataFrame objects
- axis : 0 or 'index' -> 행 방향, 1 or 'columns' -> 열 방향 
- join : {'outer', 'inner'}, 매치되는 index/columns 없을 때의 동작
    + outer : NaN 채우기
    + inner : 삭제하기
- ignore_index : index 를 무시하고 RangeIndex 로 변경
- verify_integrity : True -> 중복 데이터 있으면 오류 발생
- https://pandas.pydata.org/docs/reference/api/pandas.concat.html#pandas-concat

In [1]:
import pandas as pd
import numpy as np

In [8]:
df1 = pd.DataFrame({'A': np.arange(1, 5), 'B': list('opqr')}, index=list('abcd'))
df2 = pd.DataFrame({'A': [5, 6, 7], 'B': list('stu')}, index=list('efg'))
df3 = pd.DataFrame({'C': [10, 20, 15, 40], 'D': list('QXYZ')}, index=list('abcd'))
df4 = pd.DataFrame({'A': [1, 20, 15], 'B': list('xyz')}, index=list('abc'))

display(df1, df2, df3, df4)

,A,B
a,1,o
b,2,p
c,3,q
d,4,r


,A,B
e,5,s
f,6,t
g,7,u


,C,D
a,10,Q
b,20,X
c,15,Y
d,40,Z


,A,B
a,1,x
b,20,y
c,15,z


In [9]:
pd.concat([df1, df2])

,A,B
a,1,o
b,2,p
c,3,q
d,4,r
e,5,s
f,6,t
g,7,u


In [14]:
pd.concat([df1, df3], axis=1)

,A,B,C,D
a,1,o,10,Q
b,2,p,20,X
c,3,q,15,Y
d,4,r,40,Z


In [15]:
pd.concat([df1, df3])

,A,B,C,D
a,1.0,o,NaN,NaN
b,2.0,p,NaN,NaN
c,3.0,q,NaN,NaN
d,4.0,r,NaN,NaN
a,NaN,NaN,10.0,Q
b,NaN,NaN,20.0,X
c,NaN,NaN,15.0,Y
d,NaN,NaN,40.0,Z


In [18]:
pd.concat([df1, df4], ignore_index=True, verify_integrity=True)

,A,B
0,1,o
1,2,p
2,3,q
3,4,r
4,1,x
5,20,y
6,15,z


In [19]:
pd.concat([df1, df4], axis=1, join='outer')

,A,B,A,B
a,1,o,1.0,x
b,2,p,20.0,y
c,3,q,15.0,z
d,4,r,NaN,NaN


In [21]:
pd.concat([df1, df4], axis=1, join='inner')

,A,B,A,B
a,1,o,1,x
b,2,p,20,y
c,3,q,15,z


## pd.merge

- `pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None, left_index=False, right_index=False. ...)`
- on 에 지정된 병합 기준 또는 index 에 따라 left, right 병합
- left, right : DataFrame or Named Series
- how : {'left','right','outer','inner'}, default = 'inner'
- on : label or list, 병합 기준 지정 (columns or index level names)
- left_on, right_on : label or list, 왼쪽 / 오른쪽 병합 기준 지정
- left_index, right_index 
    + True / False 를 사용하야 index 를 병합 기준으로 사용할지 여부 지정
    + columns 가 다를 경우 True 로 지정하여야 함
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html#pandas-dataframe-merge

In [24]:
df1 = pd.DataFrame({'A': [1,20,15,5],   'B': list('XYZQ')}, index=list('abce'))
df2 = pd.DataFrame({'C': [10,20,15,40], 'B': list('XYMN')}, index=list('abcd'))
df3 = pd.DataFrame({'C': [10,20,15,40], 'D': list('XYMN')}, index=list('abcd'))
df4 = pd.DataFrame({'C': [10,20,15,40], 'D': list('QXYZ')}, index=list('abcd'))

display(df1, df2)

,A,B
a,1,X
b,20,Y
c,15,Z
e,5,Q


,C,B
a,10,X
b,20,Y
c,15,M
d,40,N


In [25]:
pd.merge(left=df1, right=df2, on='B')

,A,B,C
0,1,X,10
1,20,Y,20


In [27]:
display(df1, df2)

,A,B
a,1,X
b,20,Y
c,15,Z
e,5,Q


,C,B
a,10,X
b,20,Y
c,15,M
d,40,N


In [26]:
pd.merge(left=df1, right=df2, on='B', how='outer')

,A,B,C
0,1.0,X,10.0
1,20.0,Y,20.0
2,15.0,Z,NaN
3,5.0,Q,NaN
4,NaN,M,15.0
5,NaN,N,40.0


In [28]:
pd.merge(left=df1, right=df2, on='B', how='left')

,A,B,C
0,1,X,10.0
1,20,Y,20.0
2,15,Z,NaN
3,5,Q,NaN


In [29]:
pd.merge(left=df1, right=df2, on='B', how='right')

,A,B,C
0,1.0,X,10
1,20.0,Y,20
2,NaN,M,15
3,NaN,N,40


In [30]:
display(df1, df3)

,A,B
a,1,X
b,20,Y
c,15,Z
e,5,Q


,C,D
a,10,X
b,20,Y
c,15,M
d,40,N


In [31]:
pd.merge(left=df1, right=df3, left_on='B', right_on='D')

,A,B,C,D
0,1,X,10,X
1,20,Y,20,Y


In [32]:
display(df1.head(), df4.head())

,A,B
a,1,X
b,20,Y
c,15,Z
e,5,Q


,C,D
a,10,Q
b,20,X
c,15,Y
d,40,Z


In [33]:
pd.merge(left=df1, right=df4, left_index=True, right_index=True)

,A,B,C,D
a,1,X,10,Q
b,20,Y,20,X
c,15,Z,15,Y


# 데이터 삭제

- `x.drop(labels, axis=0, ...)`
    + labels : 한 개의 label 또는 list-like index / column labels
    + axis=0 or 'index' : 행 삭제
    + axis-1 or 'columns' : 열 삭제
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html#pandas-dataframe-drop

![nn](./images/pandas-04.png)

In [36]:
s = pd.Series([10,13,15], index=list('ABB'))
s1 = s.drop('B')
display(s, s1)

A    10
B    13
B    15
dtype: int64

A    10
dtype: int64

In [40]:
df = pd.DataFrame({'a': [10,13], 'b': [1,5], 'c': [20,5]}, index=list('AB'))
df1 = df.drop(['a', 'b'], axis=1)
display(df, df1)

,a,b,c
A,10,1,20
B,13,5,5


,c
A,20
B,5


# 데이터 추가